In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null


In [ ]:
!wget http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz

--2025-11-11 16:17:23--  http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 400446614 (382M) [application/x-gzip]
Saving to: ‘spark-3.5.1-bin-hadoop3.tgz’

spark-3.5.1-bin-had 100%[===================>] 381.90M   147KB/s    in 33m 36s 

2025-11-11 16:50:59 (194 KB/s) - ‘spark-3.5.1-bin-hadoop3.tgz’ saved [400446614/400446614]



In [ ]:
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

In [ ]:
import findspark
findspark.init()

In [ ]:
from google.colab import drive
from google.colab import userdata
drive.mount('/content/drive')

Mounted at /content/drive


In [35]:
import pyspark
from pyspark.sql import SparkSession

BASE_DIR = userdata.get('BASE_DIR')
FIRST_FILE_NAME = "customers-2000000.csv"
SECOND_FILE_NAME = "customers-100000.csv"

class ApacheSparkDB():
  def __init__(self, app_name, data_path):
    try:
      self.spark = SparkSession.builder.appName(app_name).getOrCreate()
    except Exception as e:
      print(f"Error due to: {e}")

  def read_csv(self, file_name, num_partition:int=100, num_coalesce:int=50):
    df = self.spark.read.csv(file_name, header=True, inferSchema=True)

    if num_partition > 0:
      df = df.repartition(num_partition)

    if num_coalesce > 0:
      df = df.coalesce(num_coalesce)

    print(f"The total_data is {df.count()}")

    return df

  def show_dataframe(self, df):
    return df.show(truncate=False)

  def show_schema(self, df):
    return df.printSchema()

  def query_with_sql(self, query):
    return self.spark.sql(query)


In [36]:
def list_query(table_name):
  get_dense_rank_query_main_domain = f"""
  SELECT * FROM
    (SELECT *, DENSE_RANK() OVER (PARTITION BY subs_by_month ORDER BY total_subs DESC) AS rank
      FROM
      (
        SELECT DATE_FORMAT(`Subscription Date`, 'yyyy_MM') AS subs_by_month,
                REPLACE(CASE
                    WHEN Website RLIKE '^http://' OR Website RLIKE '^https://'  THEN
                            SUBSTRING_INDEX(SUBSTRING_INDEX(Website, '/', 3), '/', -1)
                    ELSE
                            SUBSTRING_INDEX(Website, '/', 1)
                    END, 'www.', '') AS main_domain,
                COUNT(Index) AS total_subs
        FROM {table_name}
        GROUP BY subs_by_month, main_domain
        SORT BY subs_by_month ASC, total_subs DESC
        ) as temp_table
    )
    WHERE rank = 1
    SORT BY subs_by_month ASC
  """

  get_min_max_subs_date = f"""SELECT MAX(`Subscription Date`), MIN(`Subscription Date`) FROM {table_name};"""
  return [get_dense_rank_query_main_domain, get_min_max_subs_date]

## CUSTOMER_2000000

In [37]:
cust2mil_path = os.path.join(BASE_DIR, FIRST_FILE_NAME).replace('"','')
app_name = FIRST_FILE_NAME.split('.')[0].replace('-','_')

cust2mil_spark = ApacheSparkDB(app_name, cust2mil_path)

In [38]:
cust2mil_spark_df = cust2mil_spark.read_csv(cust2mil_path, num_partition=100, num_coalesce=50)


The total_data is 2000000


In [39]:
cust2mil_spark.show_dataframe(cust2mil_spark_df)

+------+---------------+----------+---------+---------------------------+-------------------+---------------------------------------------------+----------------------+---------------------+-----------------------------------+-----------------+-------------------------------+
|Index |Customer Id    |First Name|Last Name|Company                    |City               |Country                                            |Phone 1               |Phone 2              |Email                              |Subscription Date|Website                        |
+------+---------------+----------+---------+---------------------------+-------------------+---------------------------------------------------+----------------------+---------------------+-----------------------------------+-----------------+-------------------------------+
|722807|Eda152DF31AE28C|Monique   |Bass     |Lane-Bruce                 |North Karamouth    |Namibia                                            |165-346-9492x8089     |(

In [40]:
cust2mil_spark.show_schema(cust2mil_spark_df)

root
 |-- Index: integer (nullable = true)
 |-- Customer Id: string (nullable = true)
 |-- First Name: string (nullable = true)
 |-- Last Name: string (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Phone 1: string (nullable = true)
 |-- Phone 2: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Subscription Date: date (nullable = true)
 |-- Website: string (nullable = true)



In [41]:
cust2mil_spark_df.createOrReplaceTempView("cust2mil_table")

In [42]:
queries = list_query("cust2mil_table")

In [43]:
cust2mil_spark.query_with_sql(queries[0]).show()

+-------------+------------+----------+----+
|subs_by_month| main_domain|total_subs|rank|
+-------------+------------+----------+----+
|      2020_01|   kline.com|        45|   1|
|      2020_01|  coffey.com|        45|   1|
|      2020_02| rosales.com|        44|   1|
|      2020_03|  taylor.com|        46|   1|
|      2020_03| hammond.com|        46|   1|
|      2020_04|  barron.com|        46|   1|
|      2020_05| mcgrath.com|        46|   1|
|      2020_06|   nixon.com|        46|   1|
|      2020_07|     fry.com|        46|   1|
|      2020_08|  mcneil.com|        45|   1|
|      2020_09|  reeves.com|        44|   1|
|      2020_09|  norton.com|        44|   1|
|      2020_10|humphrey.com|        48|   1|
|      2020_11|  monroe.com|        43|   1|
|      2020_12|  moreno.com|        50|   1|
|      2021_01|  santos.com|        44|   1|
|      2021_02| escobar.com|        44|   1|
|      2021_03|  waller.com|        49|   1|
|      2021_04|     fry.com|        44|   1|
|      202

In [44]:
cust2mil_spark.query_with_sql(queries[1]).show()

+----------------------+----------------------+
|max(Subscription Date)|min(Subscription Date)|
+----------------------+----------------------+
|            2022-05-30|            2020-01-01|
+----------------------+----------------------+



## CUSTOMER_100000

In [45]:
cust100th_path = os.path.join(BASE_DIR, SECOND_FILE_NAME).replace('"','')
app_name = SECOND_FILE_NAME.split('.')[0].replace('-','_')

cust100th_spark = ApacheSparkDB(app_name, cust100th_path)

In [46]:
cust100th_spark_df = cust100th_spark.read_csv(cust100th_path, num_partition=100, num_coalesce=50)


The total_data is 100000


In [47]:
cust100th_spark.show_dataframe(cust100th_spark_df)

+-----+---------------+----------+----------+---------------------------+-------------------+------------------+---------------------+---------------------+--------------------------------+-----------------+---------------------------------+
|Index|Customer Id    |First Name|Last Name |Company                    |City               |Country           |Phone 1              |Phone 2              |Email                           |Subscription Date|Website                          |
+-----+---------------+----------+----------+---------------------------+-------------------+------------------+---------------------+---------------------+--------------------------------+-----------------+---------------------------------+
|60167|fcBEf068cE58eaC|Molly     |Murray    |Lambert Inc                |Fritzton           |Angola            |4513512235           |(083)851-5420x3848   |bartonamy@avila.com             |2021-11-04       |https://www.mcgee-bray.com/      |
|55742|b1046AaAc9D7FFc|Katie    

In [48]:
cust100th_spark.show_schema(cust100th_spark_df)

root
 |-- Index: integer (nullable = true)
 |-- Customer Id: string (nullable = true)
 |-- First Name: string (nullable = true)
 |-- Last Name: string (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Phone 1: string (nullable = true)
 |-- Phone 2: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Subscription Date: date (nullable = true)
 |-- Website: string (nullable = true)



In [49]:
cust100th_spark_df.createOrReplaceTempView("cust100th_spark")

In [50]:
queries = list_query("cust100th_spark")

In [51]:
cust100th_spark.query_with_sql(queries[0]).show()

+-------------+-------------+----------+----+
|subs_by_month|  main_domain|total_subs|rank|
+-------------+-------------+----------+----+
|      2020_01|    potts.com|         7|   1|
|      2020_02|    ponce.com|         6|   1|
|      2020_02|     lane.com|         6|   1|
|      2020_02|     ware.com|         6|   1|
|      2020_03|   norman.com|         6|   1|
|      2020_03|     case.com|         6|   1|
|      2020_04|     tran.com|         6|   1|
|      2020_04| delacruz.com|         6|   1|
|      2020_04|  garrett.com|         6|   1|
|      2020_05|   hayden.com|         6|   1|
|      2020_06|  elliott.com|         6|   1|
|      2020_06|  harding.com|         6|   1|
|      2020_06|     ryan.com|         6|   1|
|      2020_06|  osborne.com|         6|   1|
|      2020_06|castaneda.com|         6|   1|
|      2020_06|  hendrix.com|         6|   1|
|      2020_07|   jensen.com|         7|   1|
|      2020_08|    myers.com|         6|   1|
|      2020_08|   knight.com|     

In [52]:
cust100th_spark.query_with_sql(queries[1]).show()

+----------------------+----------------------+
|max(Subscription Date)|min(Subscription Date)|
+----------------------+----------------------+
|            2022-05-29|            2020-01-01|
+----------------------+----------------------+

